In [ ]:
# View and modify the working path
import os
from google.colab import drive

# View current working directory
print("Current Working Directory:", os.getcwd())

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
path = "/content/gdrive/My Drive/GroupProject"
os.chdir(path)

# Confirm the change
print("Working Directory:", os.getcwd())

Current Working Directory: /content/gdrive/MyDrive/GroupProject
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Working Directory: /content/gdrive/My Drive/GroupProject


In [ ]:
import gensim
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from nltk.tokenize import word_tokenize
from collections import defaultdict

# Load data
readm_balanced = pd.read_csv('gen_readm_balanced.csv')

# Load pre-trained word2vec model
pmc_word2vec_path = 'PubMed-and-PMC-w2v.bin'
word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(pmc_word2vec_path, binary=True)

# Extract and tokenize 'cleaned_text' column
texts = readm_balanced['cleaned_text'].values

# Define a dictionary for word index and a list for tokenized sequences
word_index = defaultdict(lambda: len(word_index) + 1)  # Automatically assign unique index
sequences = []

# Tokenize each text and convert it to a sequence of indices
for text in texts:
    tokens = word_tokenize(text.lower())  # Tokenize and lowercase the text
    sequence = [word_index[token] for token in tokens]
    sequences.append(sequence)

# Convert defaultdict to a regular dictionary
word_index = dict(word_index)

# Define padding length
MAX_SEQUENCE_LENGTH = 100

# Convert each sequence to a PyTorch tensor and pad all sequences
sequence_tensors = [torch.tensor(seq) for seq in sequences]
data_padded = pad_sequence(sequence_tensors, batch_first=True, padding_value=0)
data_padded = data_padded[:, :MAX_SEQUENCE_LENGTH]  # Trim to MAX_SEQUENCE_LENGTH if necessary

# Initialize embedding matrix
EMBEDDING_DIM = word2vec_model.vector_size  # Set dimension based on word2vec model
embedding_matrix = torch.zeros((len(word_index) + 1, EMBEDDING_DIM))

for word, i in word_index.items():
    if word in word2vec_model:
        embedding_matrix[i] = torch.tensor(word2vec_model[word])
    else:
        embedding_matrix[i] = torch.normal(mean=0, std=embedding_matrix.std(), size=(EMBEDDING_DIM,))

# Save padded data and embedding matrix to CSV
data_padded_df = pd.DataFrame(data_padded.numpy())
data_padded_df.to_csv('data_padded_gen.csv', index=False)

embedding_matrix_df = pd.DataFrame(embedding_matrix.numpy())
embedding_matrix_df.to_csv('embedding_matrix_gen.csv', index=False)

# Check if GPU is available and move data to GPU if it is
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move padded data and embedding matrix to GPU
data_padded = data_padded.to(device)
embedding_matrix = embedding_matrix.to(device)

In [ ]:
#Generate training and test sets
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split


# Convert data_padded and labels to tensors
data_padded = torch.tensor(data_padded, dtype=torch.long)  # Sequences of word indices
readm_balanced['gen_readm_converted'] = readm_balanced['gen_readm'].map({'positive': 1, 'negative': 0})
# Change the dtype to torch.long
labels = torch.tensor(readm_balanced['gen_readm_converted'].values, dtype=torch.long)  # Binary labels

# Define a custom Dataset class
class ReadmissionDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# Instantiate the Dataset
dataset = ReadmissionDataset(data_padded, labels)

# Split dataset into train/test using sklearn's train_test_split
train_indices, test_indices = train_test_split(range(len(dataset)), test_size=0.1, random_state=903965310, stratify=labels)

# Create subsets for train/test
train_dataset = torch.utils.data.Subset(dataset, train_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)

# Create DataLoaders for train and test sets
batch_size = 32
dataloader_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
#Create the CNN network with three kernel sizes
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import KFold

class ReadminAnalysisCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(ReadminAnalysisCNN, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

        # Multiple convolutional layers with different kernel sizes
        self.conv1d_1 = nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=1, padding=1)
        self.conv1d_2 = nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=2, padding=1)
        self.conv1d_3 = nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=3, padding=1)

        # Fully connected layer for classification
        self.fc1 = nn.Linear(128 * 3, 2)  # Softmax layer expects 2 output units for binary classification

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        # Apply each convolution followed by max pooling and concatenate results
        x1 = F.relu(self.conv1d_1(x))
        x1 = F.max_pool1d(x1, kernel_size=x1.size(2)).squeeze(2)

        x2 = F.relu(self.conv1d_2(x))
        x2 = F.max_pool1d(x2, kernel_size=x2.size(2)).squeeze(2)

        x3 = F.relu(self.conv1d_3(x))
        x3 = F.max_pool1d(x3, kernel_size=x3.size(2)).squeeze(2)

        # Concatenate all pooled features
        x = torch.cat((x1, x2, x3), dim=1)

        # Fully connected layer with softmax for probability output
        x = F.softmax(self.fc1(x), dim=1)
        return x


In [ ]:
#Train the model and save the best one

# Prepare cross-validation
k_folds = 10
kfold = KFold(n_splits=k_folds, shuffle=True)
best_model = None
best_val_accuracy = 0
vocab_size = len(word_index) + 1
embedding_dim = word2vec_model.vector_size #(usually 200 for PubMed word2vec)
# Training loop
for fold, (train_idx, val_idx) in enumerate(kfold.split(train_dataset)):
    print(f'Fold {fold + 1}/{k_folds}')

    # Create new model instance and move it to device
    CNN_model = ReadminAnalysisCNN(vocab_size, embedding_dim)

    # Define loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(CNN_model.parameters(), lr=0.001)

    # Create training and validation dataloaders for this fold
    train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
    val_sampler = torch.utils.data.SubsetRandomSampler(val_idx)
    train_loader = torch.utils.data.DataLoader(dataloader_train.dataset, sampler=train_sampler, batch_size=32)
    val_loader = torch.utils.data.DataLoader(dataloader_train.dataset, sampler=val_sampler, batch_size=32)

    # Training for multiple epochs
    for epoch in range(5):  # Adjust the number of epochs as needed
        CNN_model.train()
        running_loss = 0.0
        for notes, labels in train_loader:
            notes, labels = notes, labels

            optimizer.zero_grad()
            outputs = CNN_model(notes)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Validation step
    CNN_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for notes, labels in val_loader:
            notes, labels = notes, labels
            outputs = CNN_model(notes)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    # Calculate fold accuracy
    fold_accuracy = correct / total
    print(f'Fold {fold + 1}, Validation Accuracy: {fold_accuracy:.4f}')

    # Save the best model
    if fold_accuracy > best_val_accuracy:
        best_val_accuracy = fold_accuracy
        best_model = CNN_model.state_dict()  # Save the model's state dict

Fold 1/10
Fold 1, Validation Accuracy: 0.6207
Fold 2/10
Fold 2, Validation Accuracy: 0.6270
Fold 3/10
Fold 3, Validation Accuracy: 0.5846
Fold 4/10
Fold 4, Validation Accuracy: 0.6489
Fold 5/10
Fold 5, Validation Accuracy: 0.6254
Fold 6/10
Fold 6, Validation Accuracy: 0.6520
Fold 7/10
Fold 7, Validation Accuracy: 0.6458
Fold 8/10
Fold 8, Validation Accuracy: 0.6379
Fold 9/10
Fold 9, Validation Accuracy: 0.6207
Fold 10/10
Fold 10, Validation Accuracy: 0.6515


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Load the best model for testing
CNN_model.load_state_dict(best_model)
CNN_model.eval()

# Evaluate on test set
all_labels = []
all_predictions = []
with torch.no_grad():
    for notes, labels in dataloader_test:
        notes, labels = notes, labels
        outputs = CNN_model(notes)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)


print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Test Accuracy: {accuracy:.4f}")

Precision: 0.609
Recall: 0.831
F1 Score: 0.703
Test Accuracy: 0.6488


In [ ]:
#Use TF-IDF to encode the text. Use RandomForest to make predictions
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Load data
readm_balanced = pd.read_csv('readm_30d_balanced.csv')

# Encode labels
X_texts = readm_balanced['cleaned_text']
readm_balanced['readm_30d_converted'] = readm_balanced['readm_30d'].map({'positive': 1, 'negative': 0})
y_labels = readm_balanced['readm_30d_converted']  # Assuming 1 for positive and 0 for negative

# Split into train and test sets (90%-10%)
X_train, X_test, y_train, y_test = train_test_split(X_texts, y_labels, test_size=0.1, random_state=903965310)

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=3000)  # Start with all features
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Feature selection through GridSearch on max_features for RandomForest
param_grid = {'max_features': [1000, 3000]}  # Adjust as necessary
rf_model = RandomForestClassifier(random_state=903965310)
grid_search = GridSearchCV(rf_model, param_grid, scoring='accuracy', cv=4, n_jobs=-1)

# Train the model with GridSearchCV to find the best max_features
grid_search.fit(X_train_tfidf, y_train)
best_rf_model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_rf_model.predict(X_test_tfidf)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Best max_features:", grid_search.best_params_['max_features'])

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Test Accuracy: {accuracy:.4f}")

Best max_features: 3000
Precision: 0.630
Recall: 0.723
F1 Score: 0.673
Test Accuracy: 0.6580
